# Notebook 2 — Chunking y Formato Seq2Seq: google/mt5-small

**Objetivo:** Transformar los documentos de DisTEMIST en pares `(input_text, target_text)` adaptados a la arquitectura **Encoder-Decoder** de `google/mt5-small`.

### Especificaciones Técnicas:
1. **Límite de Encoder:** 512 tokens.
2. **Prefijo de Tarea:** `"Identifica las enfermedades mencionadas en el siguiente texto clínico: "`
3. **Chunking con Overlap:** 50 palabras de solapamiento para no cortar entidades entre límites.
4. **Formato Target:** Entidades separadas por `" [SEP] "` (o `"ninguna"` si no hay menciones).
5. **Diagnóstico de Longitud:** Análisis empírico de longitudes de input y target sin truncar silenciosamente.

## 1. Setup y Carga de Splits Crudos

In [ ]:
# Conectar Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive montado.")
except ImportError:
    print("Ejecutando fuera de Colab.")

Mounted at /content/drive
 Google Drive montado.


In [ ]:
# Dependencias
try:
    import transformers, sentencepiece
    print("Librerías listas.")
except ImportError:
    %pip install -q transformers sentencepiece datasets pandas
    print("Librerías instaladas.")

 Librerías listas.


In [ ]:
import json
import os
import re
import random
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Dataset, DatasetDict, load_from_disk
from transformers import AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Topicos")
SPLITS_DIR = BASE_DIR / "distemist_final"
raw_splits = load_from_disk(str(SPLITS_DIR / "distemist_raw_splits"))
print("Splits crudos cargados:")
print(raw_splits)

Splits crudos cargados:
DatasetDict({
    train: Dataset({
        features: ['doc_id', 'text', 'entities'],
        num_rows: 600
    })
    dev: Dataset({
        features: ['doc_id', 'text', 'entities'],
        num_rows: 75
    })
    test: Dataset({
        features: ['doc_id', 'text', 'entities'],
        num_rows: 75
    })
})


## 2. Tokenizador de `google/mt5-small` y Ventana Segura

In [ ]:
MODEL_CHECKPOINT = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"Tokenizador cargado: {MODEL_CHECKPOINT}")
print(f"Tamaño vocabulario SentencePiece: {tokenizer.vocab_size:,} tokens")

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Tokenizador cargado: google/mt5-small
Tamaño vocabulario SentencePiece: 250,100 tokens


In [ ]:
def compute_safe_window_words(tokenizer, texts, target_max_tokens=480, safety_margin=0.9):
    """
    Calcula el tamaño de ventana en palabras seguro para que ningún chunk
    exceda el límite del encoder (512 tokens), dejando margen para el prompt y tokens especiales.
    """
    ratios = []
    for text in texts:
        n_words = len(re.findall(r"\S+", text))
        n_tokens = len(tokenizer.encode(text, add_special_tokens=False))
        if n_words > 0:
            ratios.append(n_tokens / n_words)

    p95_ratio = np.percentile(ratios, 95)
    safe_window = int((target_max_tokens / p95_ratio) * safety_margin)

    print(f"Ratio tokens/palabra (p95): {p95_ratio:.2f}")
    print(f"Ventana segura calculada (window_words): {safe_window} palabras")
    return safe_window

sample_texts = [text for split_data in raw_splits.values() for text in split_data["text"]]
chosen_window_words = compute_safe_window_words(tokenizer, sample_texts, target_max_tokens=480)

Ratio tokens/palabra (p95): 2.09
Ventana segura calculada (window_words): 207 palabras


## 3. Función de Chunking con Solapamiento (Overlap) y Mapeo de Entidades

In [ ]:
OVERLAP_WORDS = 50

def chunk_record_by_words(rec, window_words=chosen_window_words, overlap_words=OVERLAP_WORDS):
    """
    Divide un documento largo en chunks solapados, preservando las entidades y remapeando offsets.
    """
    text = rec["text"]
    word_matches = list(re.finditer(r"\S+", text))

    if len(word_matches) <= window_words:
        return [rec]

    chunks = []
    start_idx = 0
    chunk_id = 0
    while start_idx < len(word_matches):
        end_idx = min(start_idx + window_words, len(word_matches))
        char_start = word_matches[start_idx].start()
        char_end = word_matches[end_idx - 1].end()

        # Filtrar entidades contenidas completamente en el chunk
        chunk_entities = [
            {**ent, "start": ent["start"] - char_start, "end": ent["end"] - char_start}
            for ent in rec["entities"]
            if ent["start"] >= char_start and ent["end"] <= char_end
        ]

        chunks.append({
            "doc_id": f"{rec['doc_id']}_chunk{chunk_id}",
            "text": text[char_start:char_end],
            "entities": chunk_entities,
        })

        if end_idx == len(word_matches):
            break
        start_idx = end_idx - overlap_words
        chunk_id += 1

    return chunks

def chunk_all_records(records, window_words=chosen_window_words, overlap_words=OVERLAP_WORDS):
    chunked = []
    for rec in records:
        chunked.extend(chunk_record_by_words(rec, window_words, overlap_words))
    return chunked

chunked_splits = {name: chunk_all_records(recs) for name, recs in raw_splits.items()}
for name, chks in chunked_splits.items():
    print(f"Split {name:<6}: {len(raw_splits[name]):>4} docs -> {len(chks):>4} chunks")

Split train :  600 docs -> 1438 chunks
Split dev   :   75 docs ->  180 chunks
Split test  :   75 docs ->  195 chunks


## 4. Construcción del Formato Sequence-to-Sequence (Input & Target)

- **`input_text`:** `"Identifica las enfermedades mencionadas en el siguiente texto clínico: " + <texto_chunk>`
- **`target_text`:** Entidades separadas por `" [SEP] "` (o `"ninguna"` si no hay menciones).

In [ ]:
ENTITY_SEP = " [SEP] "
PROMPT_PREFIX = "Identifica las enfermedades mencionadas en el siguiente texto clínico: "

def build_mt5_target(entities):
    """Construye el string objetivo respetando orden y eliminando duplicados exactos."""
    terms = [ent["text"].strip() for ent in entities if ent.get("text")]
    seen = []
    for t in terms:
        if t and t not in seen:
            seen.append(t)
    return ENTITY_SEP.join(seen) if seen else "ninguna"

def build_mt5_split(records):
    data = {"doc_id": [], "input_text": [], "target_text": []}
    for rec in records:
        data["doc_id"].append(rec["doc_id"])
        data["input_text"].append(PROMPT_PREFIX + rec["text"])
        data["target_text"].append(build_mt5_target(rec["entities"]))
    return Dataset.from_dict(data)

mt5_dataset = DatasetDict({
    split_name: build_mt5_split(records) for split_name, records in chunked_splits.items()
})

print("Dataset mT5 estructurado:")
print(mt5_dataset)
print("\nEjemplo Seq2Seq:")
print("INPUT :", mt5_dataset["train"][0]["input_text"][:140], "...")
print("TARGET:", mt5_dataset["train"][0]["target_text"])

Dataset mT5 estructurado:
DatasetDict({
    train: Dataset({
        features: ['doc_id', 'input_text', 'target_text'],
        num_rows: 1438
    })
    dev: Dataset({
        features: ['doc_id', 'input_text', 'target_text'],
        num_rows: 180
    })
    test: Dataset({
        features: ['doc_id', 'input_text', 'target_text'],
        num_rows: 195
    })
})

Ejemplo Seq2Seq:
INPUT : Identifica las enfermedades mencionadas en el siguiente texto clínico: Presentamos el caso de una paciente de 58 años con antecedentes perso ...
TARGET: lumbociática [SEP] lesiones vertebrales (en D7 y desde D9 a D12) [SEP] lesiones vertebrales múltiples de origen osteoporótico en columna dorsal [SEP] obesidad [SEP] HTA [SEP] aplastamientos vertebrales osteoporóticos [SEP] fibromialgia [SEP] hernia discal D3-D4 [SEP] hepatomegalia [SEP] disfagia


## 5. Análisis Empírico de Longitudes y Guardado en Drive

In [ ]:
# 1. Longitud del Input (Encoder)
input_lengths = [len(tokenizer.encode(item["input_text"], truncation=False)) for item in mt5_dataset["train"]]
exceed_512 = sum(1 for l in input_lengths if l > 512)

print("=" * 65)
print("  VALIDACIÓN DE LONGITUD DE ENTRADA (INPUT / ENCODER)")
print("=" * 65)
print(f"Max tokens en train input  : {max(input_lengths)}")
print(f"Media tokens en train input: {np.mean(input_lengths):.1f}")
print(f"P50 (Mediana)              : {np.percentile(input_lengths, 50):.1f}")
print(f"P95                        : {np.percentile(input_lengths, 95):.1f}")
print(f"P99                        : {np.percentile(input_lengths, 99):.1f}")
print(f"Chunks > 512 tokens        : {exceed_512} ({100*exceed_512/len(input_lengths):.2f}%)")

# 2. Longitud del Target (Decoder)
target_lengths = [len(tokenizer.encode(item["target_text"], truncation=False)) for item in mt5_dataset["train"]]
exceed_128 = sum(1 for l in target_lengths if l > 128)
exceed_192 = sum(1 for l in target_lengths if l > 192)
exceed_256 = sum(1 for l in target_lengths if l > 256)

print("\n" + "=" * 65)
print("  VALIDACIÓN DE LONGITUD DE OBJETIVO (TARGET / DECODER)")
print("=" * 65)
print(f"Max tokens en train target : {max(target_lengths)}")
print(f"Media tokens en train target: {np.mean(target_lengths):.1f}")
print(f"P50 (Mediana)              : {np.percentile(target_lengths, 50):.1f}")
print(f"P90                        : {np.percentile(target_lengths, 90):.1f}")
print(f"P95                        : {np.percentile(target_lengths, 95):.1f}")
print(f"P99                        : {np.percentile(target_lengths, 99):.1f}")
print(f"Targets > 128 tokens       : {exceed_128} ({100*exceed_128/len(target_lengths):.2f}%)")
print(f"Targets > 192 tokens       : {exceed_192} ({100*exceed_192/len(target_lengths):.2f}%)")
print(f"Targets > 256 tokens       : {exceed_256} ({100*exceed_256/len(target_lengths):.2f}%)")
print("=" * 65)

# Diagnóstico de configuración para Notebook 03
if exceed_128 == 0:
    print("MAX_TARGET_LENGTH = 128 es suficiente (0% truncamiento).")
elif exceed_192 == 0:
    print("Recomendación: Usar MAX_TARGET_LENGTH = 192 en Notebook 03.")
else:
    print("Recomendación: Usar MAX_TARGET_LENGTH = 256 en Notebook 03.")

# 3. Guardado en Drive
out_path = SPLITS_DIR / "distemist_mt5_format"
mt5_dataset.save_to_disk(str(out_path))
print(f"\nDataset mT5 persistido en: {out_path}")
print("Listo para ejecutar Notebook 03.")

  VALIDACIÓN DE LONGITUD DE ENTRADA (INPUT / ENCODER)
Max tokens en train input  : 508
Media tokens en train input: 347.6
P50 (Mediana)              : 387.0
P95                        : 450.1
P99                        : 474.6
Chunks > 512 tokens        : 0 (0.00%)

  VALIDACIÓN DE LONGITUD DE OBJETIVO (TARGET / DECODER)
Max tokens en train target : 195
Media tokens en train target: 48.2
P50 (Mediana)              : 41.0
P90                        : 98.0
P95                        : 113.1
P99                        : 160.3
Targets > 128 tokens       : 37 (2.57%)
Targets > 192 tokens       : 2 (0.14%)
Targets > 256 tokens       : 0 (0.00%)
 Recomendación: Usar MAX_TARGET_LENGTH = 256 en Notebook 03.


Saving the dataset (0/1 shards):   0%|          | 0/1438 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/180 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/195 [00:00<?, ? examples/s]


Dataset mT5 persistido en: /content/drive/MyDrive/Colab Notebooks/Topicos/distemist_final/distemist_mt5_format
Listo para ejecutar Notebook 03.
